# Module 4: Search in Azure DocumentDB (Node.js completed reference)

In [ ]:
const { MongoClient } = require("mongodb");const uri = process.env.DOCUMENTDB_CONNECTION_STRING;if (!uri) throw new Error("Set DOCUMENTDB_CONNECTION_STRING before running this notebook.");const client = new MongoClient(uri);await client.connect();const db = client.db("docdbworkshop");const collection = db.collection("workshop_content");await db.command({ ping: 1 });

In [ ]:
const docs = [  { _id: "doc-search-001", title: "DiskANN vector indexing", category: "vector", body: "Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents.", sku: "SEARCH-VEC-001", embedding: [0.92, 0.80, 0.18] },  { _id: "doc-search-002", title: "BM25 keyword search", category: "full-text", body: "Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata.", sku: "SEARCH-FTS-001", embedding: [0.20, 0.12, 0.94] },  { _id: "doc-search-003", title: "Hybrid search with RRF", category: "hybrid", body: "Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion.", sku: "SEARCH-HYB-001", embedding: [0.76, 0.70, 0.42] },  { _id: "doc-search-004", title: "RAG grounding", category: "rag", body: "Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context.", sku: "RAG-PIPE-001", embedding: [0.82, 0.74, 0.36] },  { _id: "doc-search-005", title: "Operational filtering", category: "filters", body: "Search applications often filter by status, tenant, region, stock, or category after the search stage narrows candidate documents.", sku: "SEARCH-FLT-001", embedding: [0.35, 0.30, 0.82] }];await collection.deleteMany({});await collection.insertMany(docs);await collection.countDocuments({});

In [ ]:
await db.command({ createIndexes: "workshop_content", indexes: [{ name: "idx_embedding_diskann", key: { embedding: "cosmosSearch" }, cosmosSearchOptions: { kind: "vector-diskann", dimensions: 3, similarity: "COS", maxDegree: 32, lBuild: 64 } }] });await db.command({ createSearchIndexes: "workshop_content", indexes: [{ name: "idx_body_fts", definition: { mappings: { dynamic: false, fields: { body: { type: "string" } } } } }] });

In [ ]:
const semanticQueryVector = [0.90, 0.78, 0.22];await collection.aggregate([{ $search: { cosmosSearch: { path: "embedding", vector: semanticQueryVector, k: 3 } } }, { $project: { _id: 1, title: 1, category: 1, score: { $meta: "searchScore" } } }]).toArray();

In [ ]:
await collection.aggregate([{ $search: { index: "idx_body_fts", text: { query: "BM25 ranking", path: "body" } } }, { $limit: 5 }, { $project: { _id: 1, title: 1, score: { $meta: "searchScore" } } }]).toArray();

In [ ]:
await collection.aggregate([{ $search: { index: "idx_body_fts", text: { query: "retrival augmentd genration", path: "body", fuzzy: { maxEdits: 1 } } } }, { $limit: 5 }, { $project: { _id: 1, title: 1, score: { $meta: "searchScore" } } }]).toArray();

In [ ]:
await collection.aggregate([{ $search: { index: "idx_body_fts", phrase: { query: "Reciprocal Rank Fusion", path: "body", slop: 0 } } }, { $limit: 5 }, { $project: { _id: 1, title: 1, score: { $meta: "searchScore" } } }]).toArray();

In [ ]:
function rrf(lists, k = 60, topN = 5) {  const scores = new Map();  const docsById = new Map();  for (const list of lists) {    list.forEach((doc, rank) => {      const id = doc._id.toString();      docsById.set(id, doc);      scores.set(id, (scores.get(id) ?? 0) + 1 / (k + rank + 1));    });  }  return [...scores.entries()].sort((a, b) => b[1] - a[1]).slice(0, topN).map(([id, score]) => ({ ...docsById.get(id), rrfScore: score }));}const userQuery = "semantic retrieval for rag";const queryVector = [0.84, 0.76, 0.32];const keywordHits = await collection.aggregate([{ $search: { index: "idx_body_fts", text: { query: userQuery, path: "body" } } }, { $limit: 5 }, { $project: { _id: 1, title: 1, score: { $meta: "searchScore" } } }]).toArray();const vectorHits = await collection.aggregate([{ $search: { cosmosSearch: { path: "embedding", vector: queryVector, k: 5 } } }, { $project: { _id: 1, title: 1, score: { $meta: "searchScore" } } }]).toArray();rrf([keywordHits, vectorHits]);